%md
#Ingest races.csv file
1. Read the file using Spark DataFrame reader API
2. Add Metadata Columns
    - Source File
    - Ingestion TimeStamp
3. Write to bronze delta table

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_path = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

### Step 1- Read the csv file using the DataFrame Reader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
races_schema = StructType([
    StructField("season",IntegerType()),
    StructField("round",IntegerType()),
    StructField("url",StringType()),
    StructField("raceName",StringType()),
    StructField("date",DateType()),
    StructField("circuitId",StringType())
])

In [0]:
races_df = (
    spark.read
        .format("csv")
        .option("header","true")
        .option("mode","FAILFAST")
#       .option("inferSchema","true")
        .schema(races_schema)
        .load(source_path)    
    )
races_df.printSchema()


In [0]:
display(races_df)

###Step 2- Add Metadata Columns
- Source File
- Ingestion TimeStamp



In [0]:
races_final_df = add_ingestion_metadata(races_df)

In [0]:
display(races_final_df)

### Step 3- Write to bronze delta table

In [0]:
(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)

)

In [0]:
display(spark.table(table_name))